# 🤖 Step 6 — GPT-4o Daily News Summarisation

**What this step does:**
- Takes raw GDELT headlines from Step 5
- Groups all headlines by date
- Sends each day's headlines to GPT-4o in a **few-shot setting**
- GPT-4o acts as a senior financial analyst and generates a structured 2-3 sentence summary per day
- Output is one clean summary per trading date — ready for FinBERT in Step 7

**Why GPT-4o here (thesis justification):**
> *LLMs were utilised in a few-shot setting to generate structured financial summaries from raw news headlines, overcoming the scarcity of labelled sentiment data and converting noisy unstructured text into coherent, model-ready qualitative signals.*

**Input  :** `step5a_raw_news_all_topics.csv`

**Output :** `step6_gpt4o_daily_summaries.csv`

---
### Instructions
1. Run **Cell 1** — install libraries
2. Run **Cell 2** — enter OpenAI API key and config
3. Run **Cell 3** — upload Step 5 news file
4. Run **Cell 4** — define few-shot prompt
5. Run **Cell 5** — run GPT-4o for all dates
6. Run **Cell 6** — preview results
7. Run **Cell 7** — save and download

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 1 — Install Libraries
# ─────────────────────────────────────────────────────────────────────────
!pip install openai pandas numpy -q
print('✓ Libraries ready')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 2 — Configuration
# ─────────────────────────────────────────────────────────────────────────
import os
import json
import time
import pandas as pd
import numpy as np
from openai import OpenAI
from google.colab import files

# ══════════════════════════════════════════════════════════════════════════
# ── ENTER YOUR OPENAI API KEY HERE ────────────────────────────────────────
OPENAI_API_KEY = 'your-openai-api-key-here'   # ← paste your key
# ══════════════════════════════════════════════════════════════════════════

MODEL          = 'gpt-4o'
MAX_TOKENS     = 250      # enough for a 2-3 sentence summary + JSON
TEMPERATURE    = 0.2      # low temperature = consistent factual output
SLEEP_BETWEEN  = 1.0      # seconds between API calls
MAX_HEADLINES  = 20       # top headlines per day sent to GPT-4o

# Initialise OpenAI client
client = OpenAI(api_key=OPENAI_API_KEY)

print('=' * 60)
print('  STEP 6 — GPT-4o DAILY NEWS SUMMARISATION')
print('=' * 60)
print(f'  Model          : {MODEL}')
print(f'  Temperature    : {TEMPERATURE}')
print(f'  Max tokens     : {MAX_TOKENS}')
print(f'  Max headlines  : {MAX_HEADLINES} per date')
print(f'  Sleep between  : {SLEEP_BETWEEN}s')
if OPENAI_API_KEY == 'your-openai-api-key-here':
    print()
    print('  ⚠️  Please paste your OpenAI API key above')
else:
    print('  ✓ API key set')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 3 — Upload Step 5 News File and Prepare Date-wise Headlines
# ─────────────────────────────────────────────────────────────────────────
print('Please upload step5a_raw_news_all_topics.csv ...')
uploaded = files.upload()
filename = list(uploaded.keys())[0]

# Load
news_df = pd.read_csv(filename)
print(f'\n✓ File loaded   : {filename}')
print(f'✓ Total articles: {len(news_df)}')
print(f'✓ Columns       : {list(news_df.columns)}')

# Parse published_at — handle multiple possible formats
def parse_date(series):
    # Try GDELT format first: 20250101T120000Z
    parsed = pd.to_datetime(series, format='%Y%m%dT%H%M%SZ', errors='coerce')
    # Fill any that failed with flexible parser
    mask = parsed.isna()
    if mask.any():
        parsed[mask] = pd.to_datetime(series[mask], errors='coerce')
    return parsed

news_df['published_at'] = parse_date(news_df['published_at'])
news_df['date'] = news_df['published_at'].dt.date

# Drop rows with missing date or title
before = len(news_df)
news_df = news_df.dropna(subset=['date', 'title'])
print(f'\n✓ Rows with valid date+title: {len(news_df)} (dropped {before - len(news_df)})')

# Ensure tone is numeric
news_df['tone'] = pd.to_numeric(news_df['tone'], errors='coerce')

# Sort by date and absolute tone descending
news_df['abs_tone'] = news_df['tone'].abs().fillna(0)
news_df = news_df.sort_values(['date', 'abs_tone'], ascending=[True, False])
news_df = news_df.reset_index(drop=True)

# ── Build date-wise dictionaries ───────────────────────────────────────────
# Group: date → list of headlines (sorted by strongest tone first)
date_headlines = {}
date_tone      = {}
date_count     = {}

for date_val, group in news_df.groupby('date'):
    date_headlines[date_val] = group['title'].dropna().tolist()
    date_tone[date_val]      = group['tone'].mean()
    date_count[date_val]     = len(group)

all_dates = sorted(date_headlines.keys())

# Verify dates are actual dates not column names
import datetime
all_dates = [d for d in all_dates if isinstance(d, datetime.date)]

print(f'\n✓ Unique dates with news : {len(all_dates)}')
if len(all_dates) > 0:
    print(f'✓ Date range             : {all_dates[0]} → {all_dates[-1]}')
    print(f'✓ Avg headlines per day  : {np.mean([len(date_headlines[d]) for d in all_dates]):.1f}')
    print(f'\n  Sample dates:')
    for d in all_dates[:5]:
        print(f'    {d}  :  {len(date_headlines[d])} headlines')
else:
    print('\n  ⚠️  No valid dates found — check your uploaded file')

# Cost estimate
est_tokens = len(all_dates) * (MAX_HEADLINES * 15 + MAX_TOKENS)
est_cost   = (est_tokens / 1_000_000) * 5
print(f'\n  💰 Estimated API cost : ~${est_cost:.2f} USD')
print(f'     ({len(all_dates)} dates × ~{MAX_HEADLINES * 15 + MAX_TOKENS} tokens per date)')


In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 4 — Define Few-Shot Prompt
#
# Few-shot learning: 3 labelled examples are given to GPT-4o
# covering bullish, bearish, and neutral market days.
# GPT-4o learns the expected output format from these examples
# without any model training or labelled dataset required.
#
# Output is a JSON with:
#   summary          — 2-3 sentence financial analysis
#   sentiment        — bullish / bearish / neutral
#   sentiment_score  — float -1.0 to +1.0
#   key_themes       — list of 2-4 dominant market themes
#   market_signal    — strong_buy / buy / neutral / sell / strong_sell
# ─────────────────────────────────────────────────────────────────────────

SYSTEM_PROMPT = """You are a senior equity analyst specialising in Indian stock markets.
You analyse daily news headlines and produce structured market intelligence.
Always respond with valid JSON only — no preamble, no explanation, no markdown fences.

Your JSON must always contain exactly these five keys:
  summary         : 2-3 sentence factual market analysis written in past tense
  sentiment       : exactly one of [bullish, bearish, neutral]
  sentiment_score : float between -1.0 (very bearish) and +1.0 (very bullish)
  key_themes      : list of 2-4 key market themes identified from the headlines
  market_signal   : exactly one of [strong_buy, buy, neutral, sell, strong_sell]"""

# Three few-shot examples covering bullish, bearish, neutral
FEW_SHOT_EXAMPLES = [

    # Example 1 — Bullish (RBI rate cut)
    {
        'role': 'user',
        'content': (
            'Date: 06-Feb-2025\n'
            'GDELT Avg Tone: -2.1\n'
            'Article count: 38\n'
            'Headlines:\n'
            '- RBI cuts repo rate by 25 bps to 6.25 percent in surprise move\n'
            '- Nifty surges 400 points after RBI rate cut announcement\n'
            '- FII buying resumes after three weeks of outflows\n'
            '- India inflation eases to 4.3 percent giving RBI room to act\n'
            '- Banking stocks rally led by HDFC Bank and ICICI Bank'
        )
    },
    {
        'role': 'assistant',
        'content': json.dumps({
            'summary': 'The Reserve Bank of India delivered a surprise 25 basis point rate cut to 6.25 percent, triggering a broad market rally with the Nifty gaining 400 points. Easing inflation at 4.3 percent provided the central bank room to act, while the decision reinvigorated foreign institutional buying that had been absent for three weeks. Banking stocks led the advance, reflecting improved margin expectations.',
            'sentiment': 'bullish',
            'sentiment_score': 0.82,
            'key_themes': ['RBI rate cut', 'FII inflows revival', 'banking sector rally', 'inflation relief'],
            'market_signal': 'buy'
        })
    },

    # Example 2 — Bearish (FII outflows + Fed hawkishness)
    {
        'role': 'user',
        'content': (
            'Date: 14-Jan-2025\n'
            'GDELT Avg Tone: -4.3\n'
            'Article count: 52\n'
            'Headlines:\n'
            '- FII outflows hit three month high as dollar strengthens\n'
            '- Rupee weakens past 86 against dollar amid global risk off\n'
            '- Nifty breaks key support at 23000 on heavy selling\n'
            '- US Fed signals fewer rate cuts in 2025 dampening emerging market appetite\n'
            '- India trade deficit widens to record high in December'
        )
    },
    {
        'role': 'assistant',
        'content': json.dumps({
            'summary': 'Aggressive foreign institutional selling drove the Nifty below the critical 23000 support level as the US Federal Reserve signalled a more hawkish stance, reducing appetite for emerging market assets. A strengthening dollar pushed the rupee past 86, compounding pressure on India-focused funds. A widening trade deficit added to macro headwinds, leaving market sentiment deeply negative.',
            'sentiment': 'bearish',
            'sentiment_score': -0.79,
            'key_themes': ['FII outflows', 'rupee depreciation', 'Fed hawkishness', 'trade deficit widening'],
            'market_signal': 'sell'
        })
    },

    # Example 3 — Neutral (range-bound, awaiting catalyst)
    {
        'role': 'user',
        'content': (
            'Date: 05-Mar-2025\n'
            'GDELT Avg Tone: -0.4\n'
            'Article count: 29\n'
            'Headlines:\n'
            '- India GDP growth at 6.4 percent meets market expectations\n'
            '- Nifty trades in narrow range ahead of RBI policy decision\n'
            '- Oil prices steady as OPEC holds output unchanged\n'
            '- Mixed quarterly results from India IT sector companies\n'
            '- Retail inflation data due this week watched closely by markets'
        )
    },
    {
        'role': 'assistant',
        'content': json.dumps({
            'summary': 'Markets remained range-bound as an in-line GDP print at 6.4 percent provided no fresh directional catalyst ahead of the upcoming RBI policy decision. Oil price stability removed one source of macro concern but mixed IT sector earnings kept sector rotation active. Investors adopted a wait-and-see stance pending the central bank guidance and upcoming inflation data.',
            'sentiment': 'neutral',
            'sentiment_score': -0.05,
            'key_themes': ['GDP in line with estimates', 'RBI policy awaited', 'IT sector mixed results', 'range-bound market'],
            'market_signal': 'neutral'
        })
    }
]

print('✓ Few-shot prompt defined')
print('  Examples: 3 (bullish / bearish / neutral)')
print()
print('  Output JSON fields per date:')
print('    summary         — 2-3 sentence financial analysis (→ FinBERT input)')
print('    sentiment       — bullish / bearish / neutral')
print('    sentiment_score — float -1.0 to +1.0')
print('    key_themes      — list of dominant market themes')
print('    market_signal   — strong_buy / buy / neutral / sell / strong_sell')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 5 — Run GPT-4o for All Dates
#
# For each date:
#   1. Take top MAX_HEADLINES headlines sorted by absolute tone
#      (strongest sentiment articles sent first)
#   2. Build user message with date, tone, count, headlines
#   3. Send to GPT-4o with few-shot system prompt
#   4. Parse JSON response
#   5. Store structured result
#   6. Save checkpoint every 50 dates (in case of disconnection)
# ─────────────────────────────────────────────────────────────────────────

CHECKPOINT_FILE = 'step6_checkpoint.csv'

def build_user_message(date, headlines, avg_tone, article_count):
    date_str     = pd.to_datetime(str(date)).strftime('%d-%b-%Y')
    tone_str     = f'{avg_tone:.2f}' if pd.notna(avg_tone) else 'N/A'
    headline_str = '\n'.join([f'- {h}' for h in headlines[:MAX_HEADLINES]])
    return (
        f'Date: {date_str}\n'
        f'GDELT Avg Tone: {tone_str}\n'
        f'Article count: {article_count}\n'
        f'Headlines:\n{headline_str}'
    )


def call_gpt4o(user_message, retries=3):
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        *FEW_SHOT_EXAMPLES,
        {'role': 'user', 'content': user_message}
    ]
    for attempt in range(retries):
        try:
            response = client.chat.completions.create(
                model=MODEL,
                messages=messages,
                max_tokens=MAX_TOKENS,
                temperature=TEMPERATURE,
            )
            raw = response.choices[0].message.content.strip()
            # Strip markdown fences if present
            raw = raw.replace('```json', '').replace('```', '').strip()
            return json.loads(raw)
        except json.JSONDecodeError:
            print(f'    ⚠️  JSON parse error — attempt {attempt+1}/{retries}')
            time.sleep(2)
        except Exception as e:
            err = str(e)
            if 'rate_limit' in err.lower():
                print(f'    ⚠️  Rate limit — waiting 20s...')
                time.sleep(20)
            else:
                print(f'    ⚠️  API error: {err[:80]} — attempt {attempt+1}/{retries}')
                time.sleep(5)
    return None


# ── Check for existing checkpoint ─────────────────────────────────────────
results       = []
processed     = set()
failed_dates  = []

if os.path.exists(CHECKPOINT_FILE):
    ckpt = pd.read_csv(CHECKPOINT_FILE)
    results   = ckpt.to_dict('records')
    processed = set(ckpt['date_raw'].astype(str))
    print(f'✓ Checkpoint found — resuming from {len(processed)} already processed dates')
else:
    print('  No checkpoint found — starting fresh')

# ── Run for all dates ──────────────────────────────────────────────────────
print()
print('=' * 60)
print(f'  Running GPT-4o for {len(all_dates)} dates')
print('=' * 60)

for i, date in enumerate(all_dates, 1):

    # Skip if already processed
    if str(date) in processed:
        continue

    headlines    = date_headlines.get(date, [])
    avg_tone     = date_tone.get(date, np.nan)
    article_count= date_count.get(date, 0)

    if not headlines:
        print(f'  [{i:>3}/{len(all_dates)}] {date} — no headlines, skipping')
        continue

    user_msg = build_user_message(date, headlines, avg_tone, article_count)
    parsed   = call_gpt4o(user_msg)

    if parsed:
        results.append({
            'date_raw':       str(date),
            'date':           pd.to_datetime(str(date)).strftime('%d-%b-%Y'),
            'gdelt_avg_tone': round(float(avg_tone), 4) if pd.notna(avg_tone) else None,
            'article_count':  int(article_count),
            'summary':        parsed.get('summary', ''),
            'sentiment':      parsed.get('sentiment', ''),
            'sentiment_score':parsed.get('sentiment_score', None),
            'key_themes':     ' | '.join(parsed.get('key_themes', [])),
            'market_signal':  parsed.get('market_signal', ''),
            'all_headlines':  ' | '.join(headlines[:MAX_HEADLINES]),
        })
        processed.add(str(date))

        # Print progress
        r = results[-1]
        if i <= 5 or i % 25 == 0:
            print(f'  [{i:>3}/{len(all_dates)}] {r["date"]}  '
                  f'sentiment={r["sentiment"]:<8} '
                  f'score={str(r["sentiment_score"]):>6}  '
                  f'signal={r["market_signal"]}')

    else:
        failed_dates.append(str(date))
        print(f'  [{i:>3}/{len(all_dates)}] {date} — ❌ failed after retries')

    # Save checkpoint every 50 dates
    if len(results) % 50 == 0 and results:
        pd.DataFrame(results).to_csv(CHECKPOINT_FILE, index=False)
        print(f'  💾 Checkpoint saved ({len(results)} dates done)')

    time.sleep(SLEEP_BETWEEN)

# Build final DataFrame
results_df = pd.DataFrame(results)

print()
print('=' * 60)
print('  GPT-4o COMPLETE')
print('=' * 60)
print(f'  Dates processed  : {len(results_df)}')
print(f'  Dates failed     : {len(failed_dates)}')
if failed_dates:
    print(f'  Failed dates     : {failed_dates}')
print()
print('  Sentiment distribution:')
if not results_df.empty and 'sentiment' in results_df.columns:
    print(results_df['sentiment'].value_counts().to_string())
else:
    print('  ⚠️  No results yet')
print()
print('  Market signal distribution:')
if not results_df.empty and 'market_signal' in results_df.columns:
    print(results_df['market_signal'].value_counts().to_string())
else:
    print('  ⚠️  No results yet')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 6 — Preview Results
# ─────────────────────────────────────────────────────────────────────────

print('=' * 70)
print('  STEP 6 — GPT-4o OUTPUT PREVIEW')
print('=' * 70)

print(f'\n  Total rows : {len(results_df)}')
print(f'  Columns    : {list(results_df.columns)}')

# Table preview
preview_cols = ['date', 'sentiment', 'sentiment_score', 'market_signal', 'article_count']
print(f'\n  First 5 rows:')
print(results_df[preview_cols].head(5).to_string(index=False))

# Sample summaries
print(f'\n  Sample GPT-4o summaries:')
for _, row in results_df.head(3).iterrows():
    print(f'\n  [{row["date"]}]')
    print(f'  Sentiment : {row["sentiment"]}  (score={row["sentiment_score"]:+.2f})')
    print(f'  Signal    : {row["market_signal"]}')
    print(f'  Themes    : {row["key_themes"]}')
    print(f'  Summary   : {row["summary"][:200]}')

# Score statistics
print(f'\n  Sentiment score statistics:')
scores = pd.to_numeric(results_df['sentiment_score'], errors='coerce')
print(f'    Mean   : {scores.mean():+.4f}')
print(f'    Std    : {scores.std():.4f}')
print(f'    Min    : {scores.min():+.4f}')
print(f'    Max    : {scores.max():+.4f}')

# Most bullish and bearish
print(f'\n  Most bullish day:')
best = results_df.loc[scores.idxmax()]
print(f'    {best["date"]}  score={best["sentiment_score"]:+.2f}')
print(f'    {best["summary"][:120]}')

print(f'\n  Most bearish day:')
worst = results_df.loc[scores.idxmin()]
print(f'    {worst["date"]}  score={worst["sentiment_score"]:+.2f}')
print(f'    {worst["summary"][:120]}')

# Compare GDELT tone vs GPT-4o score
print(f'\n  GDELT tone vs GPT-4o sentiment_score correlation:')
gdelt   = pd.to_numeric(results_df['gdelt_avg_tone'],   errors='coerce')
gpt4o   = pd.to_numeric(results_df['sentiment_score'],  errors='coerce')
corr    = gdelt.corr(gpt4o)
print(f'    Pearson correlation: {corr:.4f}')
print(f'    (Higher = GPT-4o agrees with GDELT, Lower = GPT-4o adds independent signal)')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 7 — Save and Download
# ─────────────────────────────────────────────────────────────────────────
OUTPUT_FILE = 'step6_gpt4o_daily_summaries.csv'

# Drop the internal date_raw column before saving
save_df = results_df.drop(columns=['date_raw'], errors='ignore')

# Sort by date
save_df['date_sort'] = pd.to_datetime(save_df['date'], dayfirst=True, errors='coerce')
save_df = save_df.sort_values('date_sort').drop(columns=['date_sort'])
save_df = save_df.reset_index(drop=True)

save_df.to_csv(OUTPUT_FILE, index=False)

import os
size = os.path.getsize(OUTPUT_FILE)

print('=' * 60)
print('  FILE SAVED')
print('=' * 60)
print(f'  File     : {OUTPUT_FILE}')
print(f'  Rows     : {len(save_df)} (one per date)')
print(f'  Columns  : {len(save_df.columns)}')
print(f'  Size     : {size:,} bytes  ({size/1024:.1f} KB)')
print()
print('  Columns:')
col_descriptions = {
    'date':            'Trading date (dd-Mon-YYYY)',
    'gdelt_avg_tone':  'Raw GDELT average tone score for the day',
    'article_count':   'Number of headlines processed by GPT-4o',
    'summary':         'GPT-4o 2-3 sentence market analysis ← FinBERT input (Step 7)',
    'sentiment':       'GPT-4o label: bullish / bearish / neutral',
    'sentiment_score': 'GPT-4o continuous score -1.0 to +1.0',
    'key_themes':      'Dominant market themes (pipe separated)',
    'market_signal':   'strong_buy / buy / neutral / sell / strong_sell',
    'all_headlines':   'Raw headlines sent to GPT-4o (pipe separated)',
}
for col in save_df.columns:
    desc = col_descriptions.get(col, '')
    print(f'    {col:<20} — {desc}')

print()
print('⬇️  Downloading...')
files.download(OUTPUT_FILE)
print(f'  ✓ {OUTPUT_FILE} downloaded')
print()
print('✅ Step 6 Complete!')
print()
print('  Next → Step 7: FinBERT Embeddings')
print('  Feed the summary column from this file into FinBERT')
print('  to generate a 768-dimensional embedding vector per date')